#  Understanding Delta Lake

### Session Agenda

- Why Delta Lake?
- Problems in Traditional Data Lakes
- Delta Lake Architecture
- ACID Transactions
- Transaction Log
- Time Travel
- Schema Enforcement
- Schema Evolution
- MERGE
- OPTIMIZE
- VACUUM
- Medallion Architecture

# Traditional Data Lake


![ChatGPT Image Jul 21, 2026, 11_26_06 PM_1784657259812.png](./ChatGPT Image Jul 21, 2026, 11_26_06 PM_1784657259812.png "ChatGPT Image Jul 21, 2026, 11_26_06 PM_1784657259812.png")

# Delta Lake: Solving Traditional Data Lake Challenges

![ChatGPT Image Jul 21, 2026, 11_26_16 PM_1784657714692.png](./ChatGPT Image Jul 21, 2026, 11_26_16 PM_1784657714692.png "ChatGPT Image Jul 21, 2026, 11_26_16 PM_1784657714692.png")

%md

# Common File Formats in Data Engineering

Data Engineers work with various file formats depending on the type of data and the use case. Each file format has its own advantages and is optimized for different workloads.

| File Format | Data Type | Storage Format | Human Readable | Common Use Cases |
|--------------|-------------------------|----------------|----------------|-----------------------------------------------|
| **CSV** | Structured | Row-Based | ✅ Yes | Data Exchange, Excel, Reporting |
| **JSON** | Semi-Structured | Row-Based | ✅ Yes | APIs, Web Applications, Configuration Files |
| **XML** | Semi-Structured | Row-Based | ✅ Yes | Enterprise Systems, SOAP APIs, Configuration Files |
| **Avro** | Structured / Semi-Structured | Row-Based (Binary) | ❌ No | Kafka, Streaming, Data Serialization |
| **ORC** | Structured | Column-Based | ❌ No | Apache Hive, Data Warehousing |
| **Parquet** | Structured | Column-Based | ❌ No | Apache Spark, Delta Lake, Data Lakes, Analytics |

---

## Key Observation

These file formats can be broadly categorized into two storage types:

- **Row-Based Storage**
  - CSV
  - JSON
  - XML
  - Avro

- **Column-Based Storage**
  - ORC
  - Parquet

Let's use the following Employee table as an example.

| Employee ID | Name | Department | Salary |
|--------------|------|------------|---------|
| 101 | Alice | HR | 50,000 |
| 102 | Bob | IT | 65,000 |
| 103 | Charlie | Finance | 70,000 |
| 104 | David | IT | 80,000 |

---

# Row-Based Storage

In row-based storage, all the values of a single row are stored together.

```text
Row 1 → 101 | Alice   | HR      | 50000
Row 2 → 102 | Bob     | IT      | 65000
Row 3 → 103 | Charlie | Finance | 70000
Row 4 → 104 | David   | IT      | 80000
```

Every row is stored one after another.

```
┌──────────────────────────────────────┐
│101│Alice  │HR      │50000            │
├──────────────────────────────────────┤
│102│Bob    │IT      │65000            │
├──────────────────────────────────────┤
│103│Charlie│Finance │70000            │
├──────────────────────────────────────┤
│104│David  │IT      │80000            │
└──────────────────────────────────────┘
```

### Best For

- Transactional systems (OLTP)
- Frequent INSERT operations
- Banking applications
- Order management systems

---

# Column-Based Storage

In column-based storage, values from the same column are stored together.

```text
Employee ID
101
102
103
104

---------------------

Name
Alice
Bob
Charlie
David

---------------------

Department
HR
IT
Finance
IT

---------------------

Salary
50000
65000
70000
80000
```

```
Employee ID        Name           Department      Salary
───────────        ───────        ──────────      ──────
101                Alice          HR              50000
102                Bob            IT              65000
103                Charlie        Finance         70000
104                David          IT              80000
```

Each column is stored separately.

### Best For

- Analytics (OLAP)
- Data Warehouses
- Apache Spark
- Data Lakes
- Business Intelligence

---

# Why Column-Based Storage is Faster?

Suppose we execute the following query:

```sql
SELECT Salary
FROM Employees;
```

## Row-Based Storage

The storage engine reads every row.

```text
101 | Alice   | HR      | 50000
102 | Bob     | IT      | 65000
103 | Charlie | Finance | 70000
104 | David   | IT      | 80000
```

Even though we only need the **Salary** column, the database must read **Employee ID, Name, Department, and Salary**.

**More data is read from disk, making queries slower.**

---

## Column-Based Storage

The storage engine reads only the **Salary** column.

```text
Salary

50000
65000
70000
80000
```

It completely skips the Employee ID, Name, and Department columns.

**Less data is read from disk, making queries much faster.**


# Why Does Delta Lake Use Parquet?

Delta Lake uses **Parquet** because Parquet is one of the **fastest and most efficient file formats** for storing and analyzing large amounts of data.

Instead of creating a new file format, Delta Lake builds on top of Parquet and adds additional features such as ACID Transactions, Time Travel, and Schema Enforcement.

---

## Why Parquet?

| Reason | Explanation |
|---------|-------------|
| **Column-Based Storage** | Stores data column by column, so Spark reads only the required columns instead of the entire row. |
| **Fast Query Performance** | Reading fewer columns means less data is read from disk, making queries much faster. |
| **High Compression** | Similar values are stored together, allowing better compression and reducing storage costs. |
| **Spark Optimized** | Parquet is the default and most optimized file format for Apache Spark. |
| **Cloud Compatible** | Works seamlessly with AWS S3, Azure ADLS, and Google Cloud Storage. |
| **Industry Standard** | Widely supported by Databricks, Spark, Snowflake, AWS, Azure, GCP, and many other platforms. |

---

## What is Missing in Parquet?

Although Parquet is fast, it does **not** provide enterprise data management features.

| Missing Feature | Added by Delta Lake |
|-----------------|---------------------|
| ACID Transactions | ✅ |
| Time Travel | ✅ |
| Transaction Log | ✅ |
| Schema Enforcement | ✅ |
| Schema Evolution | ✅ |
| MERGE, UPDATE, DELETE | ✅ |

---

## Final Concept

```text
        Delta Lake

             =

      Parquet Files
             +
     Transaction Log (_delta_log)
```

- **Parquet** provides **fast storage and query performance**.
- **Delta Lake** adds **reliability, consistency, and data management** features.

---



%md

# Delta Lake Features


---

# 1. ACID Transactions

ACID Transactions ensure that every operation performed on a Delta Table is reliable and consistent.

ACID stands for:

- Atomicity
- Consistency
- Isolation
- Durability

This feature guarantees that every transaction either completes successfully or is completely rolled back if an error occurs.

### Why is it important?

Without ACID Transactions:

- Partial writes may occur.
- Data corruption is possible.
- Concurrent updates may overwrite each other.
- Reports may produce incorrect results.

With Delta Lake, every transaction is committed safely, ensuring data consistency even when multiple users access the same table simultaneously.

---

# 2. Transaction Log (_delta_log)

The Transaction Log is the heart of Delta Lake.

Every operation performed on a Delta Table is recorded inside the `_delta_log` folder.

Examples of operations include:

- INSERT
- UPDATE
- DELETE
- MERGE
- OPTIMIZE
- VACUUM

Each successful operation creates a new version of the table.

The Transaction Log enables several advanced features such as:

- ACID Transactions
- Time Travel
- Audit History
- Rollback
- Concurrent Writes

Without the Transaction Log, a Delta Table would simply behave like a collection of Parquet files.

---

# 3. Time Travel

Time Travel allows you to query previous versions of a Delta Table.

Whenever data changes, Delta Lake creates a new version instead of permanently overwriting the existing data.

This enables you to:

- View historical versions
- Recover accidentally deleted data
- Compare old and new datasets
- Debug ETL pipelines

Time Travel is extremely useful for auditing and disaster recovery.

---

# 4. Schema Enforcement

Schema Enforcement ensures that only valid data is written into a Delta Table.

Before writing data, Delta Lake validates the schema against the existing table.

If the incoming schema does not match, the operation fails.

This prevents:

- Incorrect column names
- Wrong data types
- Missing mandatory columns
- Unexpected data corruption

Schema Enforcement improves data quality by ensuring only valid data enters the table.

---

# 5. Schema Evolution

Business requirements change over time.

New columns are frequently added to existing datasets.

Schema Evolution allows Delta Lake to automatically update the table schema when required.

Instead of recreating the table, new columns can be added safely while preserving the existing data.

This makes schema changes much easier to manage in evolving data pipelines.

---

# 6. MERGE (Upsert)

MERGE is one of the most important features in Delta Lake.

It combines multiple operations into a single command.

Using MERGE, you can:

- Insert new records
- Update existing records
- Delete records based on conditions

MERGE is widely used in:

- Incremental Loads
- Change Data Capture (CDC)
- Slowly Changing Dimensions (SCD)
- Data Warehouse ETL Pipelines

Instead of writing complex SQL logic, a single MERGE statement handles all these operations efficiently.

---

# 7. OPTIMIZE

As data grows, Delta Tables may contain thousands of small Parquet files.

Having many small files negatively impacts query performance.

The OPTIMIZE command compacts multiple small files into fewer larger files.

Benefits include:

- Faster query execution
- Reduced file scanning
- Improved Spark performance
- Better storage efficiency

OPTIMIZE is an important maintenance operation for large Delta Tables.

---

# 8. VACUUM

Delta Lake retains older files to support Time Travel.

Over time, these obsolete files consume storage space.

The VACUUM command permanently removes files that are no longer required.

Benefits include:

- Frees storage space
- Removes obsolete data files
- Improves storage management
- Keeps Delta Tables clean

VACUUM should be used carefully because deleted files cannot be recovered after the retention period.

---

# 9. Medallion Architecture

The Medallion Architecture is a best practice for organizing data pipelines using Delta Lake.

It consists of three layers:

## Bronze Layer

Stores raw data exactly as it arrives from source systems.

No transformations are applied.

---

## Silver Layer

Data is cleaned, validated, and transformed.

Common operations include:

- Removing duplicates
- Handling null values
- Standardizing formats
- Applying business rules

---

## Gold Layer

Contains business-ready, aggregated data.

This layer is optimized for:

- Dashboards
- Reporting
- Analytics
- Machine Learning

The Medallion Architecture improves data quality, governance, and maintainability while supporting scalable analytics.

---

# Summary

Delta Lake provides several powerful features that make it much more reliable than a traditional data lake.

These features include:

- ACID Transactions
- Transaction Log
- Time Travel
- Schema Enforcement
- Schema Evolution
- MERGE
- OPTIMIZE
- VACUUM
- Medallion Architecture

In the following chapters, we will explore each feature in detail through concepts, architecture diagrams, and hands-on demonstrations.

# Create and  Insert Data into the Delta Table

In [0]:
-- Create Database
CREATE DATABASE IF NOT EXISTS delta_demo;

USE delta_demo;

-- Create Delta Table
CREATE TABLE employees (
    emp_id INT,
    name STRING,
    department STRING,
    salary INT
)
USING DELTA;

-- Insert Sample Data
INSERT INTO employees VALUES
(101,'Alice','HR',50000),
(102,'Bob','IT',60000),
(103,'Charlie','Finance',70000);

SELECT * FROM employees;

###  ACID Transactions

In [0]:
UPDATE employees
SET salary = salary + 5000
WHERE emp_id = 101;

SELECT * FROM employees;

## Transaction Log (_delta_log)[](url)

Delta Table

employees/

│
├── part-00000.snappy.parquet

├── part-00001.snappy.parquet

├── part-00002.snappy.parquet
│

└── _delta_log/

      ├──00000000000000000000.json

      ├──00000000000000000001.json
      
      ├──00000000000000000002.json

In [0]:
DESCRIBE HISTORY employees;

### Time Travel

In [0]:
SELECT *
FROM employees VERSION AS OF 1;

### Schema Enforcement

In [0]:
INSERT INTO employees
VALUES
(104,'David');

### Schema Evolution

In [0]:
SELECT * FROM employees;

In [0]:
ALTER TABLE employees
ADD COLUMNS (email STRING);

In [0]:
INSERT INTO employees
VALUES
(104,'David','IT',75000,'david@company.com');

In [0]:
SELECT * FROM employees;

### MERGE (UPSERT)

In [0]:
MERGE INTO employees e
USING employee_updates s
ON e.emp_id = s.emp_id

WHEN MATCHED THEN
UPDATE SET
e.salary = s.salary

WHEN NOT MATCHED THEN
INSERT *;

### OPTIMIZE

In [0]:
OPTIMIZE employees;


### VACUUM

In [0]:
VACUUM employees RETAIN 168 HOURS;